In [1]:
import requests
import pandas as pd

print("libraries loaded ✓")

libraries loaded ✓


In [2]:
# CMS Hospital General Information
# pulling live from the CMS API

url = "https://data.cms.gov/provider-data/api/1/datastore/query/xubh-q36u/0"

params = {
    "limit": 100,  # just 100 rows to start
    "offset": 0
}

response = requests.get(url, params=params)
data = response.json()

print("Status code:", response.status_code)
print("Keys in response:", data.keys())

Status code: 200
Keys in response: dict_keys(['results', 'count', 'schema', 'query'])


In [3]:
df = pd.DataFrame(data['results'])

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

Shape: (100, 38)

Columns: ['facility_id', 'facility_name', 'address', 'citytown', 'state', 'zip_code', 'countyparish', 'telephone_number', 'hospital_type', 'hospital_ownership', 'emergency_services', 'meets_criteria_for_birthing_friendly_designation', 'hospital_overall_rating', 'hospital_overall_rating_footnote', 'mort_group_measure_count', 'count_of_facility_mort_measures', 'count_of_mort_measures_better', 'count_of_mort_measures_no_different', 'count_of_mort_measures_worse', 'mort_group_footnote', 'safety_group_measure_count', 'count_of_facility_safety_measures', 'count_of_safety_measures_better', 'count_of_safety_measures_no_different', 'count_of_safety_measures_worse', 'safety_group_footnote', 'readm_group_measure_count', 'count_of_facility_readm_measures', 'count_of_readm_measures_better', 'count_of_readm_measures_no_different', 'count_of_readm_measures_worse', 'readm_group_footnote', 'pt_exp_group_measure_count', 'count_of_facility_pt_exp_measures', 'pt_exp_group_footnote', 'te_

,facility_id,facility_name,address,citytown,state,zip_code,countyparish,telephone_number,hospital_type,hospital_ownership,...,count_of_readm_measures_better,count_of_readm_measures_no_different,count_of_readm_measures_worse,readm_group_footnote,pt_exp_group_measure_count,count_of_facility_pt_exp_measures,pt_exp_group_footnote,te_group_measure_count,count_of_facility_te_measures,te_group_footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,0,11,0,,8,8,,12,11,
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,0,8,1,,8,8,,12,12,
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,0,8,1,,8,8,,12,10,
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,7,0,,8,8,,12,7,
4,010008,CRENSHAW COMMUNITY HOSPITAL,101 HOSPITAL CIRCLE,LUVERNE,AL,36049,CRENSHAW,(334) 335-3374,Acute Care Hospitals,Proprietary,...,0,2,0,,8,Not Available,5,12,6,


In [4]:
df.to_csv("../Data/hospital_general_info.csv", index=False)

print("saved! ✓")

saved! ✓


**Testing (ignore)**

**Full Data Ingestion — All 9 Datasets**


In [8]:
import requests
import pandas as pd
import os

datasets = {
    "hospital_general_info":        "xubh-q36u",
    "readmissions_and_deaths":      "ynj2-r877",
    "medicare_spending":            "nrth-mfg3",
    "complications_and_hais":       "yc9t-dgbk",
    "healthcare_infections":        "77hc-ibv8",
    "hcahps_patient_survey":        "dgck-syfz",
    "unplanned_visits":             "632h-zaca",
    "timely_effective_care":        "yv7e-xc69",
    "payment_and_value":            "c7us-v4mf"
}

def fetch_cms_dataset(dataset_id, limit=1500):
    url = f"https://data.cms.gov/provider-data/api/1/datastore/query/{dataset_id}/0"
    all_rows = []
    offset = 0

    while True:
        params = {"limit": limit, "offset": offset}
        response = requests.get(url, params=params)
        data = response.json()

        # debug: show us what came back if something's wrong
        if "results" not in data:
            print(f"  unexpected response keys: {data.keys()}")
            print(f"  response: {data}")
            break

        rows = data["results"]
        if not rows:
            break

        all_rows.extend(rows)
        offset += limit
        print(f"  fetched {len(all_rows)} rows so far...")

        if len(rows) < limit:
            break

    return pd.DataFrame(all_rows)
print("\nall done!")


all done!


In [9]:
for name, dataset_id in datasets.items():
    print(f"\nfetching: {name}")
    df = fetch_cms_dataset(dataset_id)
    if df.empty:
        print(f"  skipping {name} — no data returned")
    else:
        filepath = f"../Data/{name}.csv"
        df.to_csv(filepath, index=False)
        print(f"saved {name} — {df.shape[0]} rows, {df.shape[1]} columns ✓")

print("\nall done!")


fetching: hospital_general_info
  fetched 1500 rows so far...
  fetched 3000 rows so far...
  fetched 4500 rows so far...
  fetched 5426 rows so far...
saved hospital_general_info — 5426 rows, 38 columns ✓

fetching: readmissions_and_deaths
  fetched 1500 rows so far...
  fetched 3000 rows so far...
  fetched 4500 rows so far...
  fetched 6000 rows so far...
  fetched 7500 rows so far...
  fetched 9000 rows so far...
  fetched 10500 rows so far...
  fetched 12000 rows so far...
  fetched 13500 rows so far...
  fetched 15000 rows so far...
  fetched 16500 rows so far...
  fetched 18000 rows so far...
  fetched 19500 rows so far...
  fetched 21000 rows so far...
  fetched 22500 rows so far...
  fetched 24000 rows so far...
  fetched 25500 rows so far...
  fetched 27000 rows so far...
  fetched 28500 rows so far...
  fetched 30000 rows so far...
  fetched 31500 rows so far...
  fetched 33000 rows so far...
  fetched 34500 rows so far...
  fetched 36000 rows so far...
  fetched 37500 rows